In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report
import joblib
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

print("=== TAHAP 1: PERSIAPAN DATA FINBERT ===")
path_folder = "/content/drive/MyDrive/NLP_Project/"

print("Memuat file X_finbert_embeddings.npy lama")
X_pub_finbert = np.load(path_folder + 'X_finbert_embeddings.npy', allow_pickle= True)
y_pub = np.load(path_folder + 'y_labels.npy', allow_pickle= True)

# 2. Potong 80:20 persis seperti skenario dosen (Out-of-Distribution)
X_train_pub_fin, X_test_pub_fin, y_train_pub, y_test_pub = train_test_split(
    X_pub_finbert, y_pub, test_size=0.20, random_state=42, stratify=y_pub
)
print(f"Data Publik Latih (80%): {X_train_pub_fin.shape[0]} baris")
print(f"Data Publik Uji Internal (20%): {X_test_pub_fin.shape[0]} baris")


print("\n=== TAHAP 2: MELATIH SVM MENGGUNAKAN FITUR FINBERT ===")
svm_finbert_ood = SVC(kernel='rbf', C=1.0, random_state=42)
svm_finbert_ood.fit(X_train_pub_fin, y_train_pub)

# SAVE MODEL (.pkl) SESUAI PERMINTAAN ANDA!
joblib.dump(svm_finbert_ood, path_folder + 'svm_finbert_OOD_model.pkl')
print("✅ Model SVM-FinBERT (Skenario OOD) tersimpan aman di Drive!")


print("HASIL UJIAN INTERNAL (20% Data Publik) - FINBERT")
y_pred_internal = svm_finbert_ood.predict(X_test_pub_fin)
print(classification_report(y_test_pub, y_pred_internal))

=== TAHAP 1: PERSIAPAN DATA FINBERT ===
Memuat file X_finbert_embeddings.npy lama
Data Publik Latih (80%): 21599 baris
Data Publik Uji Internal (20%): 5400 baris

=== TAHAP 2: MELATIH SVM MENGGUNAKAN FITUR FINBERT ===
✅ Model SVM-FinBERT (Skenario OOD) tersimpan aman di Drive!
HASIL UJIAN INTERNAL (20% Data Publik) - FINBERT
              precision    recall  f1-score   support

     Negatif       0.61      0.44      0.51      1532
      Netral       0.60      0.70      0.65      1350
     Positif       0.73      0.78      0.75      2518

    accuracy                           0.67      5400
   macro avg       0.65      0.64      0.64      5400
weighted avg       0.66      0.67      0.66      5400


=== TAHAP 3: PERSIAPAN DATA STOCKBIT (Ujian Eksternal) ===


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/NLP_qwen/INET_final.csv'

In [ ]:
print("\n=== TAHAP 3: PERSIAPAN DATA STOCKBIT (Ujian Eksternal) ===")
# Load 4.997 data Stockbit berlabel Qwen
path_qwen = "/content/drive/MyDrive/NLP_qwen/"
file_names = ['INET_labeled.csv', 'DEWA_labeled.csv', 'AADI_labeled.csv', 'BUMI_labeled.csv', 'ELSA_labeled.csv']
list_df_qwen = []
for file in file_names:
    df_temp = pd.read_csv(path_qwen + file)
    if 'content' in df_temp.columns and 'text' not in df_temp.columns:
        df_temp = df_temp.rename(columns={'content': 'text'})
    list_df_qwen.append(df_temp)

df_stockbit = pd.concat(list_df_qwen, ignore_index=True).dropna(subset=['text', 'label'])
texts_stockbit = df_stockbit['text'].astype(str).values
y_test_mandiri = df_stockbit['label'].values

# Menyiapkan GPU untuk ekstraksi
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Mesin ekstraksi: {device}")
model_name = "michaelmanurung/finbert-indonesia"
tokenizer = AutoTokenizer.from_pretrained(model_name)
finbert_model = AutoModel.from_pretrained(model_name).to(device)

def extract_embeddings(texts, batch_size=32):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Ekstraksi FinBERT Stockbit"):
        batch = texts[i : i + batch_size].tolist()
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = finbert_model(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)
    return np.vstack(all_embeddings)

# Ekstraksi 4.997 teks Stockbit menjadi angka
X_test_mandiri_fin = extract_embeddings(texts_stockbit)

# Menyimpan hasil ekstraksi Stockbit agar tidak perlu run GPU lagi ke depannya
np.save(path_folder + 'X_stockbit_finbert_embeddings.npy', X_test_mandiri_fin)
print("✅ Vektor FinBERT Stockbit tersimpan!")


print("🔥 HASIL UJIAN MANDIRI (100% Data Stockbit) - FINBERT")
y_pred_eksternal = svm_finbert_ood.predict(X_test_mandiri_fin)
print(classification_report(y_test_mandiri, y_pred_eksternal))


=== TAHAP 3: PERSIAPAN DATA STOCKBIT (Ujian Eksternal) ===
Mesin ekstraksi: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/845 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: michaelmanurung/finbert-indonesia
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Ekstraksi FinBERT Stockbit: 100%|██████████| 157/157 [30:41<00:00, 11.73s/it]


✅ Vektor FinBERT Stockbit tersimpan!
🔥 HASIL UJIAN MANDIRI (100% Data Stockbit) - FINBERT
              precision    recall  f1-score   support

     Negatif       0.30      0.21      0.25      1261
      Netral       0.25      0.28      0.26      1360
     Positif       0.49      0.54      0.52      2376

    accuracy                           0.39      4997
   macro avg       0.35      0.34      0.34      4997
weighted avg       0.38      0.39      0.38      4997

